In [ ]:
from pathlib import Path
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

## Load DELUGE spatial + timeseries data and pick a basin

In [ ]:
basins_path      = "/home/ok2907/Documents/Data/DELUGE_TEST/A_Spatial/basins.geoparquet"
inundation_path  = "/home/ok2907/Documents/Data/DELUGE_TEST/A_Spatial/inundation.geoparquet"
streamflow_path  = "/home/ok2907/Documents/Data/DELUGE_TEST/B_Timeseries/streamflow.zarr"
meteorology_path = "/home/ok2907/Documents/Data/DELUGE_TEST/B_Timeseries/meteorology.zarr"

basins     = gpd.read_parquet(basins_path)
inundation = gpd.read_parquet(inundation_path)
streamflow = xr.open_zarr(streamflow_path).load()

In [ ]:
basin_id = 2060476910 # 2060476910 # March until Danube # 2060488190 Danube after Vienna
basin_selected = basins.loc[basins.index == basin_id]
inundation_selected = inundation.loc[inundation.index.get_level_values("HYBAS_ID") == basin_id]

try:
    sf = streamflow.sel(HYBAS_ID=basin_id)
    runoff = sf["streamflow"].to_pandas()
    caravan_id = str(sf["caravan_id"].item())
except KeyError:
    runoff = pd.Series(
    data=np.nan,
    index=pd.date_range(start='1951-01-01', end='2026-05-31', freq='D'),
    name='streamflow'
    )
    caravan_id = None

flood_dates = pd.to_datetime(inundation_selected.index.get_level_values("flood_date").unique())

# Major HydroRIVERS in a padded box around the basin (so segments outside
# the basin but inside the padded plot view are also loaded).
rivers_path = "/home/ok2907/Documents/Data/HydroSheds/Rivers/HydroRIVERS_v10.gdb"
basin_geom = basin_selected.geometry.iloc[0]
basin_area = float(basin_selected["basin_area_km2"].iloc[0])

minx, miny, maxx, maxy = basin_geom.bounds
pad = max(maxx - minx, maxy - miny) * 0.15  # a bit wider than the plot padding
read_bbox = (minx - pad, miny - pad, maxx + pad, maxy + pad)

rivers = gpd.read_file(
    rivers_path,
    bbox=read_bbox,
    columns=["HYRIV_ID", "MAIN_RIV", "UPLAND_SKM", "DIS_AV_CMS", "ORD_STRA", "ORD_FLOW"],
)
rivers = rivers[rivers["UPLAND_SKM"] >= 0.05 * basin_area]
print(f"kept {len(rivers)} major river segments in bbox")

## Load daily meteorology for this basin from the zarr
`meteorology.zarr` is chunked (basin, date) with daily cadence. We select one basin, load into memory, and materialise as a pandas DataFrame indexed by date. All downstream plots access `meteo[...]` directly — no separate `daily` DataFrame is needed since the zarr is already daily and in target units.

In [ ]:
meteo_ds = xr.open_zarr(meteorology_path).sel(basin=basin_id).load()
meteo = meteo_ds.to_dataframe().drop(columns=["basin"])
meteo.index.name = "date"
meteo = meteo.sort_index()

print(meteo.shape)
print("date range:", meteo.index.min(), "->", meteo.index.max())
print("variables:", list(meteo.columns))
# meteo.head()

## Per-event inundation panels
One subplot per mapped flood event, each showing the basin outline plus that event's inundation polygon(s), with the fractional basin coverage in the title.

In [ ]:
n_events = len(flood_dates)
ncols = min(2, max(n_events, 1))
nrows = int(np.ceil(n_events / ncols))
event_colors = ["#00FF88", "#FA6C1F", "#7030A0", "#F804BB", "#8C564B"]

fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                         figsize=(4.5 * ncols, 4.5 * nrows),
                         squeeze=False)

# Scale river linewidth by upstream area so trunk stands out from tributaries.
if len(rivers):
    lw_min, lw_max = 0.4, 2.0
    up = rivers["UPLAND_SKM"].to_numpy()
    lw = lw_min + (lw_max - lw_min) * (np.log1p(up) - np.log1p(up.min())) / max(
        np.log1p(up.max()) - np.log1p(up.min()), 1e-9)
else:
    lw = None

# Pre-compute square view: pick xlim/ylim so their ratio already matches the
# latitude-adjusted aspect. This way contextily fetches tiles for the actual
# final extent (no blank area from adjustable="datalim" expanding later).
minx, miny, maxx, maxy = basin_geom.bounds
cx_lon = (minx + maxx) / 2
cy_lat = (miny + maxy) / 2
lat_aspect = 1.0 / np.cos(np.radians(cy_lat))
pad = 1.08

need_y = (maxy - miny) * pad
need_x = (maxx - minx) * pad
# Enforce x_range / y_range == lat_aspect (so subplot is square).
y_range = max(need_y, need_x / lat_aspect)
x_range = y_range * lat_aspect
xlim = (cx_lon - x_range / 2, cx_lon + x_range / 2)
ylim = (cy_lat - y_range / 2, cy_lat + y_range / 2)

data_crs = basins.crs  # EPSG:4326

# Pure grey Esri World Hillshade (not in contextily's built-in Esri bunch).
from xyzservices import TileProvider
esri_hillshade = TileProvider(
    name="Esri.WorldHillshade",
    url="https://services.arcgisonline.com/ArcGIS/rest/services/Elevation/World_Hillshade/MapServer/tile/{z}/{y}/{x}",
    attribution="",
    max_zoom=13,
)

inund_dates = inundation_selected.index.get_level_values("flood_date")
for ax, (i, fd) in zip(axes.flat, enumerate(flood_dates)):
    ev = inundation_selected.loc[inund_dates == str(fd.date())]

    # Lock the view first so contextily fetches tiles for the final bbox.
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect(lat_aspect, adjustable="box")

    cx.add_basemap(ax, crs=data_crs, source=esri_hillshade,
                   zorder=1, attribution="")

    basin_selected.plot(ax=ax, facecolor="none", edgecolor="black",
                        linewidth=1.0, zorder=2)
    if len(rivers):
        rivers.plot(ax=ax, color="#3878b8", linewidth=lw, alpha=0.9, zorder=3)
    color = event_colors[i % len(event_colors)]
    ev.plot(ax=ax, facecolor=color, edgecolor=color, alpha=0.65, zorder=4)

    pct = float(ev["inundation_basin_percentage"].sum())
    area_km2 = float(ev["area_km2"].sum())
    event_id = ", ".join(ev.index.get_level_values("event_id").unique())
    ax.set_title(
        f"{fd.date()}  ({event_id})\n"
        f"{area_km2:,.0f} km²  •  {pct:.1f}% of basin",
        fontsize=9,
    )
    # Re-apply after layer plots (geopandas can reset lims).
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Lon")
    ax.set_ylabel("Lat")
    ax.text(minx*0.995, miny*0.998, "Hillshade: ESRI World", fontdict={'size':7})

# blank leftover axes
for ax in axes.flat[n_events:]:
    ax.axis("off")

fig.suptitle(f"Basin Inundation Map (HYBAS_ID {basin_id})", weight="bold")
plt.tight_layout()
plt.savefig("/home/ok2907/Documents/Deluge/github_library/outputs/usage_notes/basin_inundation_map.png",
            dpi=300)
plt.show()

## Two-panel plot: inverted precip + runoff (if available)

In [ ]:
# Align both series on the overlapping date range
start = max(meteo.index.min(), runoff.index.min())
end   = min(meteo.index.max(), runoff.index.max())
precip_plot = meteo.loc[start:end, "total_precipitation"]
runoff_plot = runoff.loc[start:end]

infos = {
    "HYBAS_ID":      basin_id,
    "CARAVAN":       caravan_id,
    "Area":          f"{basin_selected['basin_area_km2'].item():.0f} km²",
    "Nested Basins": int(basin_selected["n_upstream"].item()),
    "Flood Events":  int(basin_selected["n_flood_events"].item()),
    "Event Sources": basin_selected["sources"].item(),
}
info_text = "\n".join(f"{k}: {v}" for k, v in infos.items())

fig, (ax_p, ax_q) = plt.subplots(
    nrows=2, ncols=1, figsize=(12, 5), sharex=True,
    gridspec_kw={"height_ratios": [2, 6]},
)

# Top: precipitation (inverted hyetograph)
ax_p.bar(precip_plot.index, precip_plot.values, width=1.0, color="black", linewidth=0)
ax_p.set_ylabel("Precip (mm/d)")
ax_p.yaxis.set_inverted(True)
ax_p.set_title(f"Precipitation & Runoff — HYBAS_ID {basin_id}", weight="bold")
ax_p.set_xlim(start, end)

# Bottom: runoff + flood event markers
ax_q.plot(runoff_plot.index, runoff_plot.values, color="tab:blue", linewidth=0.7)
for fd in flood_dates:
    if start <= fd <= end:
        ax_q.axvline(fd, color="red", linewidth=0.8, alpha=0.6)
ax_q.set_xlabel("Date")
ax_q.set_ylabel("Runoff (mm/d)", color="tab:blue")
ax_q.tick_params(axis="y", labelcolor="tab:blue")
ax_q.set_xlim(start, end)

ax_q.text(0.99, 0.97, info_text,
          transform=ax_q.transAxes, fontsize=9,
          va="top", ha="right",
          bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="grey"))

plt.subplots_adjust(hspace=0)
plt.tight_layout()
plt.savefig("/home/ok2907/Documents/Deluge/github_library/outputs/usage_notes/rainfall_runoff.png",
            dpi=300)
plt.show()

## Explore the ERA5-Land variables

In [ ]:
# Multi-panel overview of all daily ERA5-Land variables — read straight from `meteo`
# net_rad = meteo["surface_net_solar_radiation"] + meteo["surface_net_thermal_radiation"]
# wind    = np.sqrt(meteo["10m_u_component_of_wind"]**2 + meteo["10m_v_component_of_wind"]**2)
swvl_cols = [f"volumetric_soil_water_layer_{layer}" for layer in (1, 2, 3, 4)]

fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(12, 12), sharex=True)

# 1) Precip
axes[0].bar(meteo.index, meteo["total_precipitation"], width=5, color="black", linewidth=0)
axes[0].set_ylabel("P (mm/d)")
# axes[0].set_ylim(0,meteo["total_precipitation"].max()*1.05)
axes[0].yaxis.set_inverted(True)


# 2) Temperature (daily mean only)
axes[1].plot(meteo.index, meteo["2m_temperature"], color="darkorange", linewidth=0.7)
# axes[1].axhline(0, color="grey", linewidth=0.5, linestyle="--")
axes[1].set_ylabel("T2m (°C)")
# axes[1].legend(loc="upper right", fontsize=8)

# 3) PET vs net radiation
axes[2].plot(meteo.index, meteo["potential_evaporation"], color="firebrick", linewidth=0.7, label="PET")
axes[2].set_ylabel("PET (mm/d)")
# axes[2].tick_params(axis="y", labelcolor="firebrick")
# ax_r = axes[2].twinx()
# ax_r.plot(meteo.index, meteo["surface_net_solar_radiation"], color="goldenrod", linewidth=0.5, alpha=0.6, label="Net rad")
# ax_r.set_ylabel("Solar rad (W/m²)", color="goldenrod")
# ax_r.tick_params(axis="y", labelcolor="goldenrod")

# 4) Soil moisture profile
for layer, color in zip((1, 2, 3, 4), ["#b3cde3", "#6497b1", "#005b96", "#03396c"]):
    axes[3].plot(meteo.index, meteo[f"volumetric_soil_water_layer_{layer}"],
                 color=color, linewidth=0.6, label=f"L{layer}")
axes[3].set_ylabel("SWC (m³/m³)")
axes[3].legend(loc="upper right", ncol=4, fontsize=8)

# 5) Snow water equivalent
axes[4].plot(meteo.index, meteo["snow_depth_water_equivalent"], color="grey")
axes[4].set_ylabel("SWE (mm)")

# 6) Snowmelt
axes[5].plot(meteo.index, meteo["snowmelt"], color='#7d34eb')
axes[5].set_ylabel("Snowmelt (mm)")
axes[5].set_xlabel("Date")

# Flood event markers across all panels
for ax in axes:
    for fd in flood_dates:
        if meteo.index.min() <= fd <= meteo.index.max():
            ax.axvline(fd, color="red", linewidth=0.8, alpha=0.6)

fig.suptitle(f"ERA5-Land daily — HYBAS_ID {basin_id}", weight="bold")
plt.subplots_adjust(hspace=0.05)
plt.tight_layout()
plt.savefig("/home/ok2907/Documents/Deluge/github_library/outputs/usage_notes/meteo_data.png",
            dpi=300)
plt.show()

## Flood-event zooms (±60 days)
For each flood event in `flood_dates`, plot a window centered on the event:
- top: inverted daily precip (hyetograph)
- middle: cumulative precip over the window
- bottom: runoff with the event marked in red

In [ ]:
WINDOW_DAYS = 5

for fd in flood_dates:
    win_start = fd - pd.Timedelta(days=WINDOW_DAYS)
    win_end   = fd + pd.Timedelta(days=WINDOW_DAYS)

    p_win = meteo.loc[win_start:win_end, "total_precipitation"]
    q_win = runoff.loc[win_start:win_end]

    if p_win.empty or q_win.empty:
        print(f"Skipping {fd.date()} — no data in window")
        continue

    p_cum = p_win.cumsum()
    p_to_event = p_win.loc[:fd].sum()

    fig, (ax_p, ax_c, ax_q) = plt.subplots(
        nrows=3, ncols=1, figsize=(10, 6), sharex=True,
        gridspec_kw={"height_ratios": [2, 2, 4]},
    )

    # Top: inverted hyetograph
    ax_p.bar(p_win.index, p_win.values, width=1.0, color="black", linewidth=0)
    ax_p.axvline(fd, color="red", linewidth=1.0, alpha=0.7)
    ax_p.set_ylabel("P (mm/d)")
    ax_p.yaxis.set_inverted(True)
    ax_p.set_title(
        f"Flood event {fd.date()} — HYBAS_ID {basin_id}  "
        f"(P from window start to event: {p_to_event:.0f} mm)",
        weight="bold",
    )

    # Middle: accumulated precipitation (stepwise)
    ax_c.step(p_cum.index, p_cum.values, where="post",
              color="steelblue", linewidth=1.2)
    ax_c.fill_between(p_cum.index, 0, p_cum.values, step="post",
                      color="steelblue", alpha=0.2)
    ax_c.axvline(fd, color="red", linewidth=1.0, alpha=0.7)
    ax_c.set_ylabel("ΣP (mm)")

    # Bottom: runoff
    ax_q.plot(q_win.index, q_win.values, color="tab:blue", linewidth=1.0)
    ax_q.axvline(fd, color="red", linewidth=1.0, alpha=0.7,
                 label=f"Flood event {fd.date()}")
    ax_q.set_ylabel("Runoff (mm/d)", color="tab:blue")
    ax_q.tick_params(axis="y", labelcolor="tab:blue")
    ax_q.set_xlabel("Date")
    ax_q.set_xlim(win_start, win_end)
    ax_q.set_ylim(0, 5)
    ax_q.legend(loc="upper right", fontsize=8)

    plt.subplots_adjust(hspace=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Same as previous overlay plot, with an additional soil-water subplot (mean of layers 1–4)
fig, (ax_p, ax_c, ax_s, ax_q) = plt.subplots(
    nrows=4, ncols=1, figsize=(10, 9), sharex=True,
    gridspec_kw={"height_ratios": [2, 2, 3, 4]},
)

bar_width = 1.0 / max(len(flood_dates), 1)
swc_mean = meteo[swvl_cols].mean(axis=1)

for i, fd in enumerate(flood_dates):
    win_start = fd - pd.Timedelta(days=WINDOW_DAYS)
    win_end   = fd + pd.Timedelta(days=WINDOW_DAYS)

    p_win = meteo.loc[win_start:win_end, "total_precipitation"]
    q_win = runoff.loc[win_start:win_end]
    s_win = swc_mean.loc[win_start:win_end]
    if p_win.empty or q_win.empty:
        continue

    p_days = (p_win.index - fd).days.to_numpy()
    q_days = (q_win.index - fd).days.to_numpy()
    s_days = (s_win.index - fd).days.to_numpy()

    p_cum = p_win.cumsum()
    p_to_event = p_win.loc[:fd].sum()
    color = event_colors[i % len(event_colors)]
    label = f"{fd.date()}  (ΣP={p_to_event:.0f} mm)"

    ax_p.bar(p_days + (i - len(flood_dates) / 2) * bar_width,
             p_win.values, width=bar_width, color=color, linewidth=0, label=label)
    ax_c.step(p_days, p_cum.values, where="post", color=color, linewidth=1.2, label=label)
    ax_s.plot(s_days, s_win.values, color=color, linewidth=1.2, label=label)
    ax_q.plot(q_days, q_win.values, color=color, linewidth=1.0, label=label)

for ax in (ax_p, ax_c, ax_s, ax_q):
    ax.axvline(0, color="red", linewidth=1.0, alpha=0.7)

ax_p.set_ylabel("P (mm/d)")
ax_p.yaxis.set_inverted(True)
ax_p.set_title(f"All flood events — HYBAS_ID {basin_id}", weight="bold")

ax_c.set_ylabel("ΣP (mm)")

ax_s.set_ylabel("SWC mean (m³/m³)")

ax_q.set_ylabel("Runoff (mm/d)", color="tab:blue")
ax_q.tick_params(axis="y", labelcolor="tab:blue")
ax_q.set_xlabel("Days relative to mapped flood event")
ax_q.set_xlim(-WINDOW_DAYS, WINDOW_DAYS)
ax_q.set_ylim(0)
ax_q.legend(loc="upper right", fontsize=8, ncol=1)

plt.subplots_adjust(hspace=0)
plt.tight_layout()
plt.savefig("/home/ok2907/Documents/Deluge/github_library/outputs/usage_notes/flood_event_comparison.png",
            dpi=300)
plt.show()